# 🏛️ متحف مصر الكبير - مولد المشاهد الفنية
## Grand Egyptian Museum - AI Scene Generator

---

### 📋 الخطوات بالترتيب:
1. ✅ تثبيت المكتبات
2. ✅ تحميل موديل SDXL من الإنترنت مباشرة (مش محتاجة درايف)
3. ✅ تحميل LoRA المصري
4. ✅ تشغيل الـ API (FastAPI + ngrok)
5. ✅ (اختياري) واجهة Gradio

> ⚠️ **تأكدي إن Runtime نوعه GPU:** Runtime → Change runtime type → T4 GPU

---

## الخطوة 1️⃣: تثبيت المكتبات
**الوقت: 2-3 دقائق**

In [ ]:
!pip install -q diffusers transformers accelerate torch safetensors \
             fastapi uvicorn nest-asyncio pyngrok pillow gradio xformers

print("✅ تم تثبيت جميع المكتبات بنجاح!")

## الخطوة 2️⃣: استيراد المكتبات والتحقق من الـ GPU

In [ ]:
import torch
import gc
import os
import uuid
import warnings
warnings.filterwarnings('ignore')

from diffusers import StableDiffusionXLPipeline
from PIL import Image

print("✅ تم استيراد جميع المكتبات")
print(f"🖥️ Device: {'CUDA (GPU) ✅' if torch.cuda.is_available() else 'CPU ⚠️ (شغّلي GPU من Runtime)'}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_properties(0).name
    gpu_mem  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"📊 GPU: {gpu_name}")
    print(f"💾 VRAM: {gpu_mem:.1f} GB")
else:
    print("❌ مفيش GPU! اذهبي إلى: Runtime → Change runtime type → T4 GPU")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


✅ تم استيراد جميع المكتبات
🖥️ Device: CUDA (GPU) ✅
📊 GPU: Tesla T4
💾 VRAM: 15.6 GB


## الخطوة 3️⃣: تحميل نموذج SDXL من الإنترنت
**الوقت: 5-10 دقائق (أول مرة بس)**

> 💡 الموديل بيتحمل من HuggingFace مباشرة - مش محتاجة درايف خالص!

In [ ]:
print("🔄 جاري تحميل نموذج Stable Diffusion XL...")
print("⏳ هذا قد يستغرق 5-10 دقائق في أول مرة")
print("=" * 50)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionXLPipeline.from_pretrained(
    "stabilityai/stable-diffusion-xl-base-1.0",
    torch_dtype=torch.float16,
    use_safetensors=True,
    variant="fp16"
)

pipe = pipe.to(DEVICE)

# تحسينات الذاكرة
if torch.cuda.is_available():
    pipe.enable_attention_slicing()
    try:
        pipe.enable_xformers_memory_efficient_attention()
        print("✅ تم تفعيل xformers (توفير ذاكرة أفضل)")
    except:
        print("⚠️ xformers مش متاح - هيشتغل عادي")

print("=" * 50)
print("✅ تم تحميل النموذج بنجاح!")

🔄 جاري تحميل نموذج Stable Diffusion XL...
⏳ هذا قد يستغرق 5-10 دقائق في أول مرة


model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

⚠️ xformers مش متاح - هيشتغل عادي
✅ تم تحميل النموذج بنجاح!


## الخطوة 4️⃣: تحميل LoRA المصري

### 📥 خطوات تحميل ملف LoRA:

1. اذهبي إلى: **https://civitai.com/models/1704367/egyptianwalklikeone**
2. اضغطي على زرار **Download** لتحميل ملف `.safetensors`
3. في Colab، اضغطي على أيقونة **المجلد 📁** على الشمال
4. اضغطي على **Upload to session storage** 📤
5. اختاري الملف اللي حملتيه
6. استني لحد ما يظهر الملف في `/content/`
7. شغّلي الخلية الجاية ⬇️

In [ ]:
!pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 48.0 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
import glob

# ابحث عن أي ملف safetensors في /content/
lora_files = glob.glob("/content/*.safetensors")

if lora_files:
    LORA_PATH = lora_files[0]
    print(f"✅ تم العثور على ملف LoRA: {LORA_PATH}")

    print("🔄 جاري تحميل LoRA على الموديل...")
    pipe.load_lora_weights(LORA_PATH)
    print("✅ تم تحميل LoRA بنجاح! الموديل جاهز للاستخدام 🎨")
else:
    print("⏳ لم يتم العثور على ملف LoRA بعد.")
    print("")
    print("📌 خطوات التحميل:")
    print("   1. اذهبي إلى: https://civitai.com/models/1704367/egyptianwalklikeone")
    print("   2. حملي ملف .safetensors")
    print("   3. ارفعيه في Colab من أيقونة المجلد 📁 على الشمال")
    print("   4. شغّلي الخلية دي تاني")
    print("")
    print("💡 أو لو مش عايزة LoRA، الموديل شغال عادي بدونه!")

✅ تم العثور على ملف LoRA: /content/sdxl_Egyptian_WalkLikeOne.safetensors
🔄 جاري تحميل LoRA على الموديل...


Loading weights:   0%|          | 0/144 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/384 [00:00<?, ?it/s]

✅ تم تحميل LoRA بنجاح! الموديل جاهز للاستخدام 🎨


## الخطوة 5️⃣: تشغيل الـ API (FastAPI + ngrok)

### ⚙️ قبل التشغيل:
- لازم يكون عندك **ngrok account** مجاني من: https://ngrok.com
- روحي على: https://dashboard.ngrok.com/get-started/your-authtoken
- انسخي الـ **Auth Token** وحطيه في الكود بدل `YOUR_NGROK_TOKEN`

In [ ]:
!pip install pyngrok

In [ ]:
from fastapi import FastAPI
from fastapi.responses import FileResponse, JSONResponse
from pydantic import BaseModel
from pyngrok import ngrok
import nest_asyncio
import threading
import uvicorn
import time

# ====================================================
# ⬇️ حطي الـ ngrok token بتاعك هنا
NGROK_TOKEN = "32ToA7KjHIqgUJ7rKlZKJQSvRyA_3uXB9bPfjf64FQ7wJNcRe"  # ← غيّري دا
# ====================================================

# إعداد FastAPI
app = FastAPI(
    title="🏛️ Egyptian Art Generator API",
    description="API لتوليد صور فنية بأسلوب مصري فرعوني",
    version="1.0.0"
)

class PromptRequest(BaseModel):
    prompt: str
    steps: int = 30
    guidance_scale: float = 7.5

@app.get("/")
def root():
    return {"message": "🏛️ Egyptian Art Generator API is running!", "status": "ok"}

@app.get("/health")
def health():
    return {
        "status": "healthy",
        "gpu": torch.cuda.is_available(),
        "model": "SDXL Base 1.0"
    }

@app.post("/generate")
def generate(data: PromptRequest):
    try:
        print(f"🎨 جاري توليد صورة لـ: {data.prompt}")

        image = pipe(
            data.prompt,
            num_inference_steps=data.steps,
            guidance_scale=data.guidance_scale
        ).images[0]

        file_name = f"/content/generated_{uuid.uuid4()}.png"
        image.save(file_name)
        print(f"✅ تم حفظ الصورة: {file_name}")

        return FileResponse(
            file_name,
            media_type="image/png",
            filename="egyptian_art.png"
        )
    except Exception as e:
        return JSONResponse(status_code=500, content={"error": str(e)})

# تشغيل السيرفر في خيط منفصل
nest_asyncio.apply()

def start_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

thread = threading.Thread(target=start_server, daemon=True)
thread.start()
time.sleep(3)

# إعداد ngrok
ngrok.kill()
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(8000).public_url

print("=" * 55)
print(f"🌍 Public URL:    {public_url}")
print(f"📄 API Docs:      {public_url}/docs")
print(f"❤️  Health Check: {public_url}/health")
print("=" * 55)
print("")
print("🚀 مثال على الاستخدام:")
print(f'curl -X POST "{public_url}/generate" \\')
print('     -H "Content-Type: application/json" \\')
print('     -d \'{"prompt": "ancient Egyptian pharaoh, gold, hieroglyphs"}\' \\')
print('     --output image.png')

🌍 Public URL:    https://1396-34-125-128-189.ngrok-free.app
📄 API Docs:      https://1396-34-125-128-189.ngrok-free.app/docs
❤️  Health Check: https://1396-34-125-128-189.ngrok-free.app/health

🚀 مثال على الاستخدام:
curl -X POST "https://1396-34-125-128-189.ngrok-free.app/generate" \
     -H "Content-Type: application/json" \
     -d '{"prompt": "ancient Egyptian pharaoh, gold, hieroglyphs"}' \
     --output image.png


## الخطوة 6️⃣: (اختياري) واجهة Gradio

> لو عايزة تجربي الموديل بواجهة بصرية بدون API

In [ ]:
import gradio as gr

def generate_image(prompt, steps, guidance):
    if not prompt.strip():
        return None
    print(f"🎨 توليد: {prompt}")
    img = pipe(
        prompt,
        num_inference_steps=int(steps),
        guidance_scale=float(guidance)
    ).images[0]
    return img

with gr.Blocks(title="Egyptian Art Generator") as demo:
    gr.Markdown("# 🏛️ Egyptian Art Generator")
    gr.Markdown("اكتبي وصف المشهد بالإنجليزية وسيتم توليد صورة فنية بأسلوب مصري فرعوني")

    with gr.Row():
        with gr.Column():
            prompt_input = gr.Textbox(
                label="📝 الوصف (Prompt)",
                placeholder="ancient Egyptian queen, gold jewelry, hieroglyphs background, divine light",
                lines=3
            )
            with gr.Row():
                steps_slider  = gr.Slider(10, 50, value=30, step=5, label="⚙️ عدد الخطوات")
                guidance_slider = gr.Slider(1, 15, value=7.5, step=0.5, label="🎯 Guidance Scale")
            generate_btn = gr.Button("🎨 توليد الصورة", variant="primary")

        with gr.Column():
            output_image = gr.Image(label="🖼️ الصورة المولّدة", type="pil")

    gr.Examples(
        examples=[
            ["ancient Egyptian pharaoh, gold crown, hieroglyphs, epic lighting", 30, 7.5],
            ["Egyptian goddess Isis with wings, temple background, golden hour", 30, 7.5],
            ["Nile river at sunset, pyramids, ancient boats, warm colors", 30, 7.5],
        ],
        inputs=[prompt_input, steps_slider, guidance_slider]
    )

    generate_btn.click(
        fn=generate_image,
        inputs=[prompt_input, steps_slider, guidance_slider],
        outputs=output_image
    )

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://484d7ec1a4e84b288f.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## 💡 نصائح للـ Prompts

اكتبي الوصف بالإنجليزي وضيفي كلمات زي:
- `ancient Egyptian style` - أسلوب مصري قديم
- `pharaoh, hieroglyphs` - فرعون وهيروغليفية
- `golden hour lighting` - إضاءة ذهبية
- `ultra detailed, 4k` - تفاصيل عالية
- `digital art, concept art` - رسم رقمي

## 🔗 كيف تستخدمي الـ API من كودك

```python
import requests

url = "YOUR_PUBLIC_URL/generate"  # الـ URL اللي ظهرلك في الخطوة 5

response = requests.post(url, json={
    "prompt": "ancient Egyptian pharaoh, gold crown, hieroglyphs",
    "steps": 30,
    "guidance_scale": 7.5
})

# حفظ الصورة
with open("my_image.png", "wb") as f:
    f.write(response.content)
```